In [11]:
from qiskit_nature.second_q.drivers import PySCFDriver
from pyscf import geomopt, scf
import time

bad_guess_co2 = "C 0 0 0; O 0 1.16 0; O 1.16 0 0"
bad_guess_h2o = "O 0.0000 0.0000 0.0000; H 0.0000 0.7572 -0.5865; H 0.0000 -0.7572 -0.5865"

# 1. Initialize the driver and build the molecule
driver = PySCFDriver(atom=bad_guess_h2o, basis="sto-3g")
problem = driver.run()

# 2. Extract the underlying PySCF molecule object
pyscf_mol = driver._mol

# 3. Create a Mean-Field electronic structure method (RHF) using that molecule
#    CRITICAL STEP: The optimizer requires this mf object!
mf = scf.RHF(pyscf_mol)

# 4. Now run the optimizer using 'mf' instead of 'pyscf_mol'
optimized_mol = geomopt.optimize(mf)

from IPython.display import clear_output

# 1. Run your optimizer (this will generate the huge text printout)
optimized_mol = geomopt.optimize(mf)

# 2. IMMEDIATELY WIPE THE SCREEN BLANK
clear_output(wait=True)
time.sleep(0.1)

# 3. Now extract and print only what you care about
raw_coords = optimized_mol.atom_coords(unit="Angstrom")
elements = [optimized_mol.atom_symbol(i) for i in range(optimized_mol.natm)]

print("--- Raw Elements ---")
print(elements)

print("\n--- Raw NumPy Matrix ---")
print(raw_coords)

import numpy as np

# 1. Choose your zero-threshold cut-off
threshold = 1e-5

# 2. Force tiny values to zero, keep everything else exactly as it was calculated
filtered_coords = np.where(np.abs(raw_coords) < threshold, 0.0, raw_coords)

# Force 4 decimal places for everything, turning 0. into 0.0000
np.set_printoptions(formatter={'float_kind': lambda x: f"{x: .4f}"})

print("--- Filtered Coordinates ---")
print(filtered_coords)

print("--- Uniform Atom List ---")
for element, coord in zip(elements, filtered_coords):
    print(f"{element:<2} {coord[0]:.4f}  {coord[1]:.4f}  {coord[2]:.4f}")

--- Raw Elements ---
['O', 'H', 'H']

--- Raw NumPy Matrix ---
[[ 0.0000  0.0000  0.0329]
 [-0.0000  0.7581 -0.6030]
 [ 0.0000 -0.7581 -0.6030]]
--- Filtered Coordinates ---
[[ 0.0000  0.0000  0.0329]
 [ 0.0000  0.7581 -0.6030]
 [ 0.0000 -0.7581 -0.6030]]
--- Uniform Atom List ---
O  0.0000  0.0000  0.0329
H  0.0000  0.7581  -0.6030
H  0.0000  -0.7581  -0.6030
